<a href="https://colab.research.google.com/github/Deivs117/Proyecto_Analisis_BioBART/blob/main/biobart_inferencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 BioBART: Inferencia en Vivo — Demo de Sustentación

**Materia:** Procesamiento de Datos Secuenciales — Módulo PLN  
**Modelo:** [`GanjinZero/biobart-large`](https://huggingface.co/GanjinZero/biobart-large) (Encoder-Decoder pre-entrenado en PubMed)  

---

## Objetivo del notebook

Demostrar en dos tareas concretas cómo BioBART **genera** texto biomédico coherente a partir de una entrada:

| # | Tarea | Descripción |
|---|-------|-------------|
| 1 | **Resumen (Summarization)** | Condensa un abstract clínico complejo en un resumen ejecutivo |
| 2 | **Diálogo / QA Médico** | Responde una pregunta clínica dado un contexto de historia médica |

---

## Flujo general del Encoder-Decoder en inferencia

```
Texto de entrada (tokens)
        │
        ▼
  ┌─────────────┐       Tokenización: convierte texto → IDs numéricos
  │  Tokenizer  │       (BPE / SentencePiece según el vocabulario de BioBART)
  └──────┬──────┘
         │ input_ids, attention_mask
         ▼
  ┌─────────────┐       Encoder: Self-Attention bidireccional sobre toda la entrada
  │   Encoder   │  →    Produce "memoria contextual" para el Decoder
  └──────┬──────┘
         │ encoder_hidden_states  (K y V para Cross-Attention)
         ▼
  ┌─────────────┐       Decoder: genera token a token (autorregresivo)
  │   Decoder   │  →    Masked Self-Attn → Cross-Attn → FFN → Linear → Softmax
  └──────┬──────┘
         │ secuencia de IDs generados
         ▼
  ┌─────────────┐
  │  Tokenizer  │  →    decode() convierte IDs → texto legible
  └─────────────┘
```


---
## Celda 1 — Importaciones y verificación del entorno

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")  # Silencia warnings de versiones menores

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── Verificar disponibilidad de GPU ──────────────────────────────────────────
# Si hay GPU disponible (CUDA), la inferencia será significativamente más rápida.
# En entornos de demo sin GPU, el modelo corre igualmente en CPU (más lento).
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Dispositivo de cómputo: {DEVICE.upper()}")
print(f"🔧  Versión de PyTorch   : {torch.__version__}")

# ── Mostrar memoria de GPU si está disponible ─────────────────────────────────
if DEVICE == "cuda":
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🧠  VRAM disponible       : {mem_gb:.1f} GB")

---
## Celda 2 — Carga del tokenizador y del modelo BioBART

> **¿Qué es `AutoTokenizer` y `AutoModelForSeq2SeqLM`?**  
> Son clases "auto-detect" de Hugging Face que leen la tarjeta del modelo en el Hub y
> seleccionan automáticamente el tokenizador y la clase de modelo correcta (en este caso,
> `BartTokenizerFast` y `BartForConditionalGeneration`).  
> `ForSeq2SeqLM` indica que el modelo tiene cabeza de lenguaje para generación secuencia-a-secuencia.


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CARGA DEL MODELO
# Usamos 'biobart-large' (400 M parámetros) para mayor calidad de generación.
# Si la memoria es limitada, cambia a 'biobart-base' (140 M parámetros).
# ──────────────────────────────────────────────────────────────────────────────
MODEL_NAME = "GanjinZero/biobart-large"   # cambiar a biobart-base si RAM < 8 GB

print(f"⏳ Cargando tokenizador desde: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✅ Tokenizador listo.")

print(f"⏳ Cargando modelo (puede tardar 1-3 minutos la primera vez) ...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Mover modelo al dispositivo correcto (CPU o CUDA)
model = model.to(DEVICE)

# Poner modelo en modo evaluación (desactiva Dropout y BatchNorm de entrenamiento)
model.eval()

num_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ Modelo listo — {num_params:.0f} M parámetros — ejecutando en {DEVICE.upper()}")

---
## ═══════════════════════════════════════════════
## TAREA 1 — Resumen de Abstract Clínico (Summarization)
## ═══════════════════════════════════════════════

### ¿Por qué resumir textos médicos?
Los profesionales de salud y los investigadores deben procesar cientos de artículos científicos.  
Un modelo generativo como BioBART puede condensar un abstract de 200 palabras en 2-3 oraciones  
clave, ahorrando tiempo crítico en entornos clínicos.  

**BERT no puede hacer esto**: BERT produce embeddings de comprensión, pero no tiene un decoder  
causal capaz de *redactar* un texto nuevo. BioBART sí lo hace.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# TEXTO DE ENTRADA: abstract clínico complejo (simulado para la demo)
# ──────────────────────────────────────────────────────────────────────────────
abstract_clinico = """
Background: Sepsis-associated acute kidney injury (SA-AKI) remains a leading cause of
morbidity and mortality in critically ill patients admitted to intensive care units (ICUs).
The pathophysiology involves complex interplay between systemic inflammatory response syndrome
(SIRS), endothelial dysfunction, microvascular thrombosis, and mitochondrial dysfunction in
proximal tubular cells. Despite advances in renal replacement therapy (RRT), mortality rates
for SA-AKI exceed 40-60% in mechanically ventilated patients.

Methods: We conducted a multicenter, randomized, double-blind, placebo-controlled trial
enrolling 847 adult patients (>=18 years) with confirmed septic shock (Sepsis-3 criteria) and
evidence of AKI stage >=2 (KDIGO 2012). Patients received intravenous infusion of recombinant
alkaline phosphatase (recAP, 1.6 mg/kg/day) or saline placebo for 3 consecutive days.
The primary endpoint was 28-day all-cause mortality. Secondary endpoints included renal
recovery at day 28, duration of RRT, ICU length of stay, and plasma biomarkers (IL-6, IL-10,
LPS-binding protein, creatinine, NGAL).

Results: Treatment with recAP significantly reduced 28-day mortality (hazard ratio [HR] 0.68;
95% CI 0.51-0.89; p=0.005) compared to placebo. Renal recovery at day 28 was achieved in
61.4% of recAP patients versus 44.7% in the placebo group (p<0.001). Plasma IL-6 levels
showed a 47% reduction from baseline at 72 hours in the treatment arm. No significant
differences were observed in ICU length of stay or adverse event profiles between groups.

Conclusions: Recombinant alkaline phosphatase significantly improves survival and renal
recovery in patients with SA-AKI. The anti-inflammatory and endotoxin-detoxifying mechanisms
of recAP represent a promising therapeutic target in septic shock-associated organ failure.
Further studies are warranted to establish optimal dosing regimens and long-term outcomes.
"""

print("📄 Abstract de entrada (primeras 300 chars):")
print(abstract_clinico[:300], "...")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# TOKENIZACIÓN
# ──────────────────────────────────────────────────────────────────────────────
# max_length=512: límite del contexto del encoder de BioBART (como en BERT-large).
# truncation=True: si el texto supera 512 tokens, se trunca (no causa error).
# return_tensors="pt": devuelve tensores de PyTorch (pt), requerido por el modelo.
# .to(DEVICE): mueve los tensores al mismo dispositivo que el modelo.
inputs_resumen = tokenizer(
    abstract_clinico,
    max_length=512,
    truncation=True,
    return_tensors="pt"
).to(DEVICE)

num_tokens = inputs_resumen["input_ids"].shape[1]
print(f"🔢 Número de tokens de entrada: {num_tokens}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# GENERACIÓN — TAREA 1: RESUMEN
#
# Hiperparámetros de model.generate() explicados:
#
#  max_new_tokens=120
#    Cantidad máxima de tokens que el decoder puede GENERAR (no contar la entrada).
#    Ajustar según la longitud esperada del resumen.
#    Muy alto → riesgo de repetición; muy bajo → resumen incompleto.
#
#  min_length=40
#    Longitud mínima del output. Evita que el modelo emita EOS demasiado pronto
#    produciendo un resumen trivialmente corto.
#
#  num_beams=4  (Beam Search)
#    En lugar de elegir 1 token (greedy), mantiene las 4 hipótesis más probables
#    en paralelo y elige la secuencia con mayor probabilidad conjunta al finalizar.
#    PROBLEMA del Greedy Search: siempre elige el token más probable en cada paso
#    → puede quedar atrapado en máximos locales y repetir frases.
#    Beam Search mitiga esto explorando múltiples caminos.
#
#  early_stopping=True
#    Detiene Beam Search cuando TODOS los beams producen EOS.
#    Evita continuar después del cierre natural de la oración.
#
#  no_repeat_ngram_size=3
#    Penaliza la repetición de cualquier 3-grama ya generado.
#    Solución directa al problema de bucles lexicales del Greedy Search.
#
#  length_penalty=1.5
#    Penaliza secuencias cortas para favorecer resúmenes más completos.
#    Valores > 1.0 premian longitud; < 1.0 favorecen brevedad.
# ──────────────────────────────────────────────────────────────────────────────

print("⚙️  Generando resumen con Beam Search (num_beams=4) ...")

with torch.no_grad():  # Desactiva gradientes → menor uso de memoria en inferencia
    ids_resumen = model.generate(
        **inputs_resumen,
        max_new_tokens=120,        # Resumen de aprox. 3-4 oraciones
        min_length=40,             # Nunca terminar antes de 40 tokens
        num_beams=4,               # Beam Search: 4 hipótesis simultáneas
        early_stopping=True,       # Parar cuando todos los beams alcancen EOS
        no_repeat_ngram_size=3,    # Prohibir repetición de trigramas
        length_penalty=1.5,        # Favorecer salidas más largas y completas
    )

# ── Decodificación: IDs → texto ───────────────────────────────────────────────
# skip_special_tokens=True elimina tokens como <s>, </s>, <pad> del output final.
resumen_generado = tokenizer.decode(ids_resumen[0], skip_special_tokens=True)

print("\n" + "-" * 70)
print("📝 RESUMEN GENERADO POR BioBART:")
print("-" * 70)
print(resumen_generado)
print("-" * 70)
print(f"   Tokens generados: {ids_resumen.shape[1]}")

---
## ═══════════════════════════════════════════════
## TAREA 2 — Diálogo / QA Médico (Medical Question Answering)
## ═══════════════════════════════════════════════

### ¿Qué es QA Médico con modelos generativos?
A diferencia de los modelos extractivos (que *localizan* un fragmento del contexto que responde
la pregunta, como hace BERT en SQuAD), BioBART **genera** una respuesta nueva en lenguaje natural,
sintetizando la información del contexto clínico.

El formato estándar para este tipo de tarea es:
```
question: <pregunta>  context: <historia clínica / nota médica>
```
El modelo aprende durante el fine-tuning a asociar este patrón con respuestas coherentes.

> **Diferencia clave con Summarization:**  
> - En resumen, el decoder condensa libremente el contenido.  
> - En QA, el decoder debe *responder con precisión* a la pregunta dentro del contexto dado.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CONTEXTO CLÍNICO (nota de admisión a UCI, simulada para la demo)
# ──────────────────────────────────────────────────────────────────────────────
contexto_clinico = """
Patient: 67-year-old male admitted to the ICU with a 4-day history of fever (39.2C),
productive cough with purulent sputum, and progressive dyspnea. Past medical history
includes type 2 diabetes mellitus (HbA1c 9.1%), chronic kidney disease stage 3 (eGFR 42
mL/min/1.73m2), and hypertension managed with amlodipine 10 mg/day.

Physical exam: SpO2 84% on room air, respiratory rate 28 breaths/min, blood pressure
88/54 mmHg requiring vasopressor support (norepinephrine 0.15 mcg/kg/min). Chest X-ray
revealed bilateral infiltrates consistent with ARDS.

Labs: WBC 18,400/uL (bands 22%), procalcitonin 42 ng/mL, lactate 4.8 mmol/L,
creatinine 3.1 mg/dL (baseline 1.4 mg/dL), troponin I 0.8 ng/mL.

Blood cultures x2 drawn; empiric therapy initiated with piperacillin-tazobactam
4.5 g IV q6h and vancomycin (dosed per renal function). Patient intubated and placed
on lung-protective ventilation (tidal volume 6 mL/kg IBW, PEEP 10 cmH2O, FiO2 0.8).
"""

# ── Pregunta formulada al modelo ──────────────────────────────────────────────
pregunta = "What are the main risk factors for poor prognosis in this patient?"

# ── Construcción del prompt en formato estándar de QA para BioBART ────────────
# El prefijo 'question: ... context: ...' es el formato aprendido durante fine-tuning
# en datasets biomédicos como BioASQ y MedQA.
prompt_qa = f"question: {pregunta}  context: {contexto_clinico}"

print("❓ Pregunta:", pregunta)
print("\n📋 Contexto clínico (primeras 200 chars):")
print(contexto_clinico[:200], "...")

In [ ]:
# ── Tokenización del prompt QA ────────────────────────────────────────────────
inputs_qa = tokenizer(
    prompt_qa,
    max_length=512,
    truncation=True,
    return_tensors="pt"
).to(DEVICE)

print(f"🔢 Tokens de entrada (prompt QA): {inputs_qa['input_ids'].shape[1]}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# GENERACIÓN — TAREA 2: QA MÉDICO
#
# Para QA usamos muestreo estocástico (do_sample=True) en lugar de Beam Search puro,
# para obtener respuestas más naturales y variadas. Combinamos con Top-P y temperatura.
#
#  do_sample=True
#    Activa el muestreo probabilístico. En vez de tomar el argmax (greedy),
#    se MUESTREA de la distribución de probabilidad → mayor diversidad.
#    Importante para respuestas que suenan naturales, no mecánicas.
#
#  temperature=0.7
#    Controla la "aleatoriedad" o entropía de la distribución softmax:
#      T < 1.0 → distribución más "puntiaguda", respuestas más seguras/predecibles
#      T = 1.0 → distribución original del modelo
#      T > 1.0 → distribución más plana, más creatividad pero menos coherencia clínica
#    Para QA médico, T=0.7 equilibra coherencia y fluidez.
#
#  top_k=50
#    Top-K Sampling: en cada paso, sólo los 50 tokens más probables
#    se mantienen como candidatos. El resto se descarta (prob = 0).
#    Evita que el modelo elija tokens rarísimos e incoherentes.
#
#  top_p=0.92  (Nucleus Sampling)
#    Se selecciona el conjunto mínimo de tokens cuya probabilidad acumulada >= 0.92.
#    Este conjunto varía en tamaño según el contexto:
#      Si el modelo está seguro → nucleus pequeño (pocas opciones buenas)
#      Si el modelo duda → nucleus grande (más diversidad)
#    Top-P > Top-K en calidad de generación de lenguaje abierto.
#
#  repetition_penalty=1.3
#    Divide la probabilidad de tokens ya generados por este factor.
#    Complementa no_repeat_ngram_size de forma continua (no binaria).
# ──────────────────────────────────────────────────────────────────────────────

print("⚙️  Generando respuesta con Nucleus Sampling (do_sample=True, top_p=0.92) ...")

with torch.no_grad():
    ids_qa = model.generate(
        **inputs_qa,
        max_new_tokens=150,          # Respuesta de aprox. 4-5 oraciones
        do_sample=True,              # Muestreo estocástico (no greedy)
        temperature=0.7,             # Controla entropía: balanceado para QA clínico
        top_k=50,                    # Top-K: descartar tokens improbables
        top_p=0.92,                  # Nucleus Sampling: mínimo conjunto >= 92% prob.
        repetition_penalty=1.3,      # Penaliza repetición de tokens ya generados
        no_repeat_ngram_size=3,      # Prohibir trigramas repetidos
    )

# ── Decodificación ────────────────────────────────────────────────────────────
respuesta_generada = tokenizer.decode(ids_qa[0], skip_special_tokens=True)

print("\n" + "-" * 70)
print("💬 RESPUESTA GENERADA POR BioBART:")
print("-" * 70)
print(respuesta_generada)
print("-" * 70)
print(f"   Tokens generados: {ids_qa.shape[1]}")

---
## Celda de cierre — Resumen de hiperparámetros y conclusiones

| Hiperparámetro | Tarea 1 (Resumen) | Tarea 2 (QA) | Efecto |
|---|---|---|---|
| `max_new_tokens` | 120 | 150 | Longitud máxima del output generado |
| `num_beams` | 4 | — | Beam Search: explora múltiples hipótesis |
| `do_sample` | False | True | Greedy/Beam vs. muestreo probabilístico |
| `temperature` | — | 0.7 | Ajusta aleatoriedad (1.0=original, <1=más determinista) |
| `top_k` | — | 50 | Limita candidatos a los K más probables |
| `top_p` | — | 0.92 | Nucleus Sampling: acumulado >= P% |
| `no_repeat_ngram_size` | 3 | 3 | Prohíbe repetición de N-gramas |
| `length_penalty` | 1.5 | — | Beam Search: premia secuencias más largas |
| `repetition_penalty` | — | 1.3 | Penaliza tokens ya generados |

---
### ¿Por qué evitar el Greedy Search puro?

El **Greedy Search** (argmax en cada paso) es propenso a:
1. **Repeticiones**: el token más probable en el paso *t* puede serlo también en *t+1*, creando bucles.
2. **Máximos locales**: una elección buena localmente puede cerrar caminos globalmente mejores.
3. **Falta de diversidad léxica**: textos médicos repetitivos generan desconfianza clínica.

Las alternativas implementadas aquí (Beam Search + Nucleus Sampling) resuelven estos problemas
manteniendo coherencia biomédica — imprescindible en aplicaciones clínicas reales.

In [ ]:
# ── Comparativa rápida: Greedy vs Beam Search (para mostrar en sustentación) ──
print("🔬 COMPARATIVA: Greedy Search vs Beam Search en Tarea 1")
print("="*70)

# Greedy Search (num_beams=1, do_sample=False) — comportamiento por defecto
with torch.no_grad():
    ids_greedy = model.generate(
        **inputs_resumen,
        max_new_tokens=80,
        num_beams=1,       # Sin Beam Search: puro argmax en cada paso
        do_sample=False,   # Sin muestreo
    )

texto_greedy = tokenizer.decode(ids_greedy[0], skip_special_tokens=True)

print("\n[GREEDY SEARCH — Argmax en cada paso]:")
print(texto_greedy)

print("\n" + "-"*70)
print("[BEAM SEARCH (num_beams=4) — Exploración de hipótesis]:")
print(resumen_generado)
print("\n→ Observar diferencias en fluidez, completitud y repetición léxica.")